# Wake Word "Letícia" para HA Voice PE (v23 — microWakeWord)

## Correções v23 vs v22

| # | Correção | Impacto |
|---|----------|---------|
| 22a | Substituída `rhasspy/piper-sample-generator/generate_samples.py` por `piper` CLI direto | Fix `CalledProcessError exit 2` — script rhasspy não aceita `.onnx`, `--cuda` não existe |
| 22b | `piper-tts` adicionado ao Etapa 1a | Fix `piper: command not found` |
| 22c | `sys.executable` em todos os subprocessos | Garante Python correto do kernel |
| 22d | ThreadPoolExecutor para geração paralela | Geração ~4x mais rápida com 4 workers |
| 20  | Auto-Resample (22050Hz → 16kHz) no Piper | Fix `Clip does not have the correct sample rate` |

## Por que v23 e não continuar o v21?

| | **v21 (openWakeWord)** | **v23 (microWakeWord)** |
|---|---|---|
| Engine | Wyoming add-on (servidor) | ESP32-S3 no chip |
| Compatível com | M5Stack Atom Echo, ESP32-S3-BOX | **HA Voice PE** |
| Output | `leticia.tflite` (Wyoming) | `stream_state_internal_quant.tflite` + JSON |
| Deploy | `/share/openwakeword/` | ESPHome YAML + OTA |

## Instruções
1. **GPU T4 ativa** (Ambiente de execução → Alterar tipo → T4 GPU)
2. Execute **Etapa 1a** → Clique **Reiniciar sessão** quando solicitado
3. Execute **Etapa 1b em diante** (pode usar "Executar tudo a partir daqui")
4. Os arquivos `leticia_mww.tflite` e `leticia_mww.json` serão baixados automaticamente

> **Tempo total estimado:** ~1-2 horas

---

## Etapa 1a: Instalação das dependências

**Instale e depois clique em "Reiniciar sessão" quando aparecer o botão.**

In [ ]:
import subprocess, sys, os

print("=" * 60)
print("  ETAPA 1a: Instalação das dependências microWakeWord (v23)")
print("=" * 60)

# Clonar microWakeWord
if not os.path.exists("microWakeWord"):
    print("\n[1a] Clonando microWakeWord...")
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/kahrendt/microWakeWord"], check=True)
    print("  ✅ microWakeWord clonado")
else:
    print("  ✅ microWakeWord já existe")

def pip_install(pkgs, desc=""):
    label = desc or pkgs[0]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ❌ {label}\n{r.stderr[-400:]}")
    else:
        print(f"  ✅ {label}")
    return r.returncode == 0

# Dependências na ordem correta
pip_install(["git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version"],
            "pymicro-features")
pip_install(["git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"],
            "audio-metadata")
pip_install(["-e", "./microWakeWord"], "microwakeword (local)")
pip_install(["torch==2.4.0+cu121", "torchaudio==2.4.0+cu121",
             "--index-url", "https://download.pytorch.org/whl/cu121"],
            "torch + torchaudio (cu121)")

# FIX v23: piper-tts fornece o CLI `piper` para geração de amostras
pip_install(["piper-tts", "piper-phonemize-cross==1.2.1"],
            "piper-tts + piper-phonemize-cross (FIX 22b)")

pip_install(["datasets", "scipy", "tqdm", "pyyaml"], "datasets / scipy / tqdm / pyyaml")

print()
print("=" * 60)
print("  ⚠️  REINICIE O RUNTIME AGORA")
print("  Runtime → Reiniciar sessão (ou Ctrl+M .)")
print("  Depois execute Etapa 1b em diante")
print("=" * 60)

## Etapa 1b: Verificação do ambiente

In [ ]:
import importlib, subprocess, sys, shutil, torch, os

print("=" * 60)
print("  ETAPA 1b: Verificação do ambiente (v23)")
print("=" * 60)

all_ok = True

# Pacotes Python
for pkg, attr in [("torch", "__version__"), ("torchaudio", "__version__"),
                  ("microwakeword", "__version__"), ("datasets", "__version__"),
                  ("scipy", "version"), ("pymicro_features", None)]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, attr, "ok") if attr else "ok"
        print(f"  ✅ {pkg}: {ver}")
    except ImportError:
        print(f"  ❌ {pkg}: NÃO instalado — execute Etapa 1a e reinicie")
        all_ok = False

# CLI piper (FIX 22b)
piper_path = shutil.which("piper")
if piper_path:
    print(f"  ✅ piper CLI: {piper_path}")
else:
    print(f"  ❌ piper CLI: NÃO encontrado — execute Etapa 1a e reinicie")
    all_ok = False

# GPU
cuda = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda else "N/A"
print(f"\n  {'✅' if cuda else '❌'} GPU: {gpu_name if cuda else 'NÃO disponível — Ative T4 GPU'}")
if not cuda:
    all_ok = False

# microWakeWord clonado
mww_path = os.path.exists("microWakeWord")
print(f"  {'✅' if mww_path else '❌'} microWakeWord clonado: {mww_path}")
if not mww_path:
    all_ok = False

if not all_ok:
    raise RuntimeError("Corrija os erros acima antes de continuar")
print("\n[OK] ETAPA 1b CONCLUÍDA!")

## Etapa 2: Piper TTS — Download de vozes pt_BR

Baixa a voz `pt_BR-faber-medium.onnx` para geração de amostras.
A voz gera áudio em 22050Hz — o FIX 20 na Etapa 3 converte para 16kHz.

In [ ]:
import os, subprocess

print("=" * 60)
print("  ETAPA 2: Download de vozes Piper pt_BR")
print("=" * 60)

# FIX v23: Sem clone do rhasspy/piper-sample-generator — usa piper CLI diretamente
os.makedirs("piper_voices_ptbr", exist_ok=True)

HF_BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main"
VOICES = [
    ("pt_BR-faber-medium", "pt/pt_BR/faber/medium"),
    ("pt_BR-edresson-low",  "pt/pt_BR/edresson/low"),
]

for name, path in VOICES:
    for ext in [".onnx", ".onnx.json"]:
        dest = f"piper_voices_ptbr/{name}{ext}"
        if not os.path.exists(dest):
            url = f"{HF_BASE}/{path}/{name}{ext}"
            print(f"\n[2] Baixando {name}{ext}...")
            subprocess.run(["wget", "-q", "--show-progress", "-O", dest, url], check=True)
            size_mb = os.path.getsize(dest) / 1024**2
            print(f"  ✅ {name}{ext} ({size_mb:.1f} MB)")
        else:
            size_mb = os.path.getsize(dest) / 1024**2
            print(f"  ✅ {name}{ext} já existe ({size_mb:.1f} MB)")

# Testar piper com faber-medium
from IPython.display import Audio, display
test_out = "/tmp/test_leticia_faber.wav"
r = subprocess.run(
    ["piper", "--model", "piper_voices_ptbr/pt_BR-faber-medium.onnx",
     "--output_file", test_out],
    input="letícia", capture_output=True, text=True
)
if os.path.exists(test_out):
    print("\nTeste de pronúncia — ouça:")
    display(Audio(test_out, autoplay=False))
else:
    print(f"  ❌ Piper falhou: {r.stderr[:300]}")

print("\n[OK] ETAPA 2 CONCLUÍDA!")

## Etapa 3: Gerar amostras TTS com piper CLI

**FIX 22a + 22c + 22d**: Usa `piper` CLI diretamente (aceita `.onnx`, sem `--cuda`),
com `sys.executable` correto e geração paralela com `ThreadPoolExecutor`.

**FIX 20**: Piper faber-medium gera em 22050Hz → resample automático para 16kHz.

In [ ]:
import os, subprocess, uuid, random, time, torchaudio, torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

print("=" * 60)
print("  ETAPA 3: Geração de amostras TTS (v23 — piper CLI direto)")
print("=" * 60)

# ── Configuração ────────────────────────────────────────────────
TARGET_WORD = "letícia"
N_SAMPLES   = 1000
SAMPLES_DIR = "positive_samples"
N_WORKERS   = 4

VOICES = [
    "piper_voices_ptbr/pt_BR-faber-medium.onnx",
    "piper_voices_ptbr/pt_BR-edresson-low.onnx",
]
# Filtrar vozes disponíveis
VOICES = [v for v in VOICES if os.path.exists(v)]
print(f"  Vozes disponíveis: {VOICES}")

LENGTH_SCALES = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25]
NOISE_SCALES  = [0.50, 0.60, 0.667, 0.70, 0.80, 0.90, 0.98]
NOISE_WS      = [0.50, 0.60, 0.70, 0.80, 0.90, 0.98]

os.makedirs(SAMPLES_DIR, exist_ok=True)
existing = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]

# ── Geração paralela com piper CLI (FIX 22a: sem generate_samples.py) ──
def gen_one_clip(args):
    word, voice, out_path = args
    try:
        r = subprocess.run(
            # FIX 22a: piper CLI aceita .onnx nativamente, sem --cuda
            ["piper",
             "--model",        voice,
             "--output_file",  out_path,
             "--length-scale", str(random.choice(LENGTH_SCALES)),
             "--noise-scale",  str(random.choice(NOISE_SCALES)),
             "--noise-w",      str(random.choice(NOISE_WS))],
            input=word,
            capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out_path) and os.path.getsize(out_path) > 0
    except Exception:
        return False

if len(existing) >= N_SAMPLES:
    print(f"  ✅ {len(existing)} amostras já existem — pulando geração")
else:
    needed = N_SAMPLES - len(existing)
    print(f"\n[3a] Gerando {needed} amostras para \"{TARGET_WORD}\" com {N_WORKERS} workers...")

    tasks = [
        (TARGET_WORD, random.choice(VOICES), os.path.join(SAMPLES_DIR, f"{uuid.uuid4().hex}.wav"))
        for _ in range(needed)
    ]

    count = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(gen_one_clip, t): t for t in tasks}
        pbar = tqdm(total=needed, desc="Gerando clips", unit="clip")
        for future in as_completed(futures):
            if future.result():
                count += 1
            pbar.update(1)
        pbar.close()

    elapsed = time.time() - t0
    generated = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
    print(f"  ✅ {len(generated)} amostras geradas em {elapsed/60:.1f} min")

# ── FIX 20: Resample 22050Hz → 16kHz ────────────────────────────
print("\n[FIX 20] Verificando sample rate (faber-medium = 22050Hz)...")
wavs = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
bad = []
for f in wavs:
    try:
        info = torchaudio.info(f"{SAMPLES_DIR}/{f}")
        if info.sample_rate != 16000:
            bad.append((f, info.sample_rate))
    except Exception:
        pass

if bad:
    print(f"  ⚠️  {len(bad)} clips em {bad[0][1]}Hz — convertendo para 16kHz...")
    for fname, sr in tqdm(bad, desc="Resample 16kHz"):
        path = f"{SAMPLES_DIR}/{fname}"
        wf, _ = torchaudio.load(path)
        if wf.shape[0] > 1:
            wf = wf.mean(dim=0, keepdim=True)
        wf = torchaudio.transforms.Resample(sr, 16000)(wf)
        torchaudio.save(path, wf, 16000)
    print(f"  ✅ {len(bad)} clips convertidos para 16kHz")
else:
    print(f"  ✅ Todos os {len(wavs)} clips já estão em 16kHz")

# ── Validação final ──────────────────────────────────────────────
valid = 0
for f in wavs:
    try:
        if torchaudio.info(f"{SAMPLES_DIR}/{f}").sample_rate == 16000:
            valid += 1
    except Exception:
        pass

print(f"\n  ✅ {valid}/{len(wavs)} amostras válidas (16kHz)")
if valid < N_SAMPLES * 0.9:
    raise RuntimeError(f"Apenas {valid}/{N_SAMPLES} amostras válidas. Verifique a instalação do piper.")

print("\n[OK] ETAPA 3 CONCLUÍDA!")

## Etapa 4: Download dos datasets negativos

Spectrogramas pré-processados do HuggingFace (kahrendt/microwakeword).
**Total ~9.7 GB** — pode levar 15-30 minutos.

In [ ]:
import os, subprocess, zipfile

print("=" * 60)
print("  ETAPA 4: Download datasets negativos")
print("=" * 60)

NEG_BASE = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main"
NEG_DATASETS = [
    ("dinner_party",      "negative_datasets/dinner_party"),
    ("dinner_party_eval", "negative_datasets/dinner_party_eval"),
    ("speech",            "negative_datasets/speech"),
    ("no_speech",         "negative_datasets/no_speech"),
]

os.makedirs("negative_datasets", exist_ok=True)

for name, dest_dir in NEG_DATASETS:
    if os.path.exists(dest_dir) and len(os.listdir(dest_dir)) > 0:
        print(f"  ✅ {name}: já existe ({len(os.listdir(dest_dir))} arquivos)")
        continue
    zip_path = f"{name}.zip"
    if not os.path.exists(zip_path):
        print(f"\n  [{name}] Baixando...")
        subprocess.run(["wget", "-q", "--show-progress", "-c",
                        f"{NEG_BASE}/{name}.zip", "-O", zip_path], check=True)
    print(f"  [{name}] Extraindo...")
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    n = len(os.listdir(dest_dir))
    print(f"  ✅ {name}: {n} arquivos")

print("\n  Resumo:")
for _, d in NEG_DATASETS:
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    status = "✅" if n > 0 else "❌"
    print(f"    {status} {os.path.basename(d)}: {n} arquivos")

print("\n[OK] ETAPA 4 CONCLUÍDA!")

## Etapa 5: Configuração do treinamento (YAML)

In [ ]:
import yaml

print("=" * 60)
print("  ETAPA 5: Criando training_parameters.yaml")
print("=" * 60)

config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/wakeword",

    "features": [
        {
            "features_dir": "generated_augmented_features",
            "sampling_weight": 2.0,
            "penalty_weight": 1.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/speech",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/no_speech",
            "sampling_weight": 5.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party_eval",
            "sampling_weight": 0.0,
            "penalty_weight": 0.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],

    "training_steps":        [50000],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates":        [0.001],
    "batch_size":            128,

    "time_mask_max_size": [0],
    "time_mask_count":    [0],
    "freq_mask_max_size": [0],
    "freq_mask_count":    [0],

    "eval_step_interval":  1000,
    "clip_duration_ms":    1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

with open("training_parameters.yaml", "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("training_parameters.yaml criado:")
print(yaml.dump(config, default_flow_style=False, allow_unicode=True))
print("[OK] ETAPA 5 CONCLUÍDA!")

## Etapa 6: Gerar features + Treinar modelo

**~1-2 horas** com GPU T4. O treinamento pode ser retomado se interrompido (`--restore_checkpoint 1`).

In [ ]:
import os, subprocess, sys

print("=" * 60)
print("  ETAPA 6: Feature generation + Treinamento")
print("=" * 60)

# ── 6a: Gerar features aumentados das amostras positivas ────────
if not os.path.exists("generated_augmented_features") or \
   len(os.listdir("generated_augmented_features")) == 0:
    print("\n[6a] Gerando features das amostras positivas...")
    os.makedirs("generated_augmented_features", exist_ok=True)

    # FIX 22c: sys.executable garante o Python correto do kernel
    gen_cmds = [
        [sys.executable, "-m", "microwakeword.generate",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
        [sys.executable, "-m", "microwakeword.generate_features",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
        [sys.executable, "microWakeWord/microwakeword/generate.py",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
        [sys.executable, "microWakeWord/microwakeword/generate_features.py",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
    ]

    generated = False
    for cmd in gen_cmds:
        module_or_script = cmd[1] if cmd[1].startswith("-m") else cmd[1]
        print(f"  Tentando: {' '.join(cmd[:3])}...")
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  ✅ Features gerados com: {' '.join(cmd[:3])}")
            generated = True
            break
        else:
            stderr_preview = r.stderr[:300].strip() if r.stderr else r.stdout[:300].strip()
            print(f"  ⚠️  Falhou: {stderr_preview}")

    if not generated:
        # Listar módulos disponíveis no microwakeword
        print("  ❌ Não foi possível gerar features. Módulos disponíveis:")
        r = subprocess.run([sys.executable, "-c",
                            "import microwakeword; import os, inspect; "
                            "print(os.path.dirname(inspect.getfile(microwakeword)))"],
                           capture_output=True, text=True)
        mww_dir = r.stdout.strip()
        if mww_dir and os.path.exists(mww_dir):
            py_files = [f for f in os.listdir(mww_dir) if f.endswith(".py")]
            print(f"  Arquivos .py em {mww_dir}: {py_files}")
        raise RuntimeError("Geração de features falhou. Veja os logs acima.")
else:
    n = len(os.listdir("generated_augmented_features"))
    print(f"  ✅ generated_augmented_features já existe ({n} arquivos)")

# ── 6b: Treinar ─────────────────────────────────────────────────
print("\n[6b] Treinando modelo microWakeWord (50k steps)...")
print("  Isso pode levar 1-2 horas com T4 GPU.\n")

# FIX 22c: sys.executable em vez de "python"
train_cmd = [
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

subprocess.run(train_cmd, check=True)

# ── Verificar output ─────────────────────────────────────────────
import glob
TFLITE_PATH = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

if not os.path.exists(TFLITE_PATH):
    print("\n⚠️  Procurando tflite em trained_models/...")
    candidates = glob.glob("trained_models/**/*.tflite", recursive=True)
    if candidates:
        TFLITE_PATH = sorted(candidates, key=os.path.getmtime)[-1]
        print(f"  Encontrado: {TFLITE_PATH}")

if os.path.exists(TFLITE_PATH):
    size_kb = os.path.getsize(TFLITE_PATH) / 1024
    print(f"\n  ✅ Modelo: {TFLITE_PATH} ({size_kb:.1f} KB)")
else:
    raise FileNotFoundError("Modelo .tflite não encontrado. Veja os logs acima.")

print("\n[OK] ETAPA 6 CONCLUÍDA!")

## Etapa 7: Empacotar, criar JSON e baixar

Gera o JSON manifest para ESPHome e baixa os dois arquivos.

In [ ]:
import os, json, shutil, glob
from google.colab import files

print("=" * 60)
print("  ETAPA 7: Empacotamento e download")
print("=" * 60)

# Localizar tflite
tflite_src = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
if not os.path.exists(tflite_src):
    candidates = glob.glob("trained_models/**/*.tflite", recursive=True)
    if candidates:
        tflite_src = sorted(candidates, key=os.path.getmtime)[-1]
        print(f"  Usando: {tflite_src}")
    else:
        raise FileNotFoundError("Nenhum .tflite encontrado. Execute Etapa 6 primeiro.")

os.makedirs("output", exist_ok=True)
tflite_dest = "output/leticia_mww.tflite"
shutil.copy(tflite_src, tflite_dest)

size_kb = os.path.getsize(tflite_dest) / 1024
print(f"  ✅ {tflite_dest} ({size_kb:.1f} KB)")

# JSON manifest para ESPHome micro_wake_word
# IMPORTANTE: Após subir o tflite para o GitHub Releases,
# atualize a URL abaixo antes de usar o JSON no ESPHome
GITHUB_RELEASE_URL = (
    "https://github.com/visaodeempresa/ha-wakeword-leticia"
    "/releases/download/v1.0-mww/leticia_mww.tflite"
)

manifest = {
    "type": "micro",
    "wake_word": "Letícia",
    "author": "Maycon Willian",
    "website": "https://github.com/visaodeempresa/ha-wakeword-leticia",
    "model": GITHUB_RELEASE_URL,
    "trained_languages": ["pt"],
    "version": 1,
    "micro": {
        "probability_cutoff": 0.5,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 26080,
        "minimum_esphome_version": "2024.7.0",
    },
}

json_dest = "output/leticia_mww.json"
with open(json_dest, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"  ✅ {json_dest}")

print("""
╔══════════════════════════════════════════════════════════╗
║         PRÓXIMOS PASSOS — HA Voice PE                   ║
╚══════════════════════════════════════════════════════════╝

1. CRIAR RELEASE no GitHub:
   https://github.com/visaodeempresa/ha-wakeword-leticia/releases/new
   Tag: v1.0-mww
   Upload: leticia_mww.tflite
   Copie a URL pública do arquivo

2. HOSPEDAR o JSON:
   Edite leticia_mww.json → atualize "model": com a URL do Release
   Suba leticia_mww.json também no Release
   Copie a URL pública do JSON

3. EDITAR ESPHome do HA Voice PE:
   Configurações → ESPHome → (dispositivo) → Editar
   Adicione em micro_wake_word → models:

   - model: https://URL_DO_JSON/leticia_mww.json
     id: leticia

4. INSTALAR via OTA
   (HA empurra automaticamente para o dispositivo)

5. No HA Voice PE: Wake word → Letícia ✅
""")

# Download automático
print("Baixando arquivos...")
for dest in [tflite_dest, json_dest]:
    try:
        files.download(dest)
    except Exception as e:
        print(f"  ⚠️  Download automático falhou para {dest}.")
        print(f"     Baixe manualmente pela aba de Arquivos 📁 à esquerda.")

print("\n[OK] ETAPA 7 CONCLUÍDA!")